# 06 — Run the confirmatory layer (CPU, minutes)

## THE BLIND BREAKS HERE

Every amendment A2–A8 certifies in writing that no targeted-factor value has been
inspected. Running section 8 ends that, permanently. Do not run it until notebooks 01
and 03 are complete and A7/A8 are committed — **no further amendment can honestly be
filed afterwards.**

Runs H1–H4, ε_G (null-quantile primary, with the CI-of-mean and fixed-0.05
sensitivities co-reported), the headroom-normalized `G̃`, the saturation gate, the flip
count, and the figures, over the three-cell realized grid.

**Setup:** CPU is fine. Internet **On**.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

In [ ]:
# Restore prior probe/calibration OUTPUTS (not checkpoints) so a timed-out
# session continues instead of recomputing.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 5. Pre-flight — the realized grid must match A7 exactly

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/probe-capacity-invariance")
from src.probes.hypotheses import discover_cells, EXPECTED_CELLS, PRE_VERDICT_EXCLUSIONS
cells = sorted(p.name for p in discover_cells("/kaggle/working/probe-capacity-invariance/results/probes"))
print("realized grid:", cells)
print("expected     :", sorted(EXPECTED_CELLS))
print("excluded     :", list(PRE_VERDICT_EXCLUSIONS))
assert set(cells) == set(EXPECTED_CELLS), "grid mismatch -- resolve BEFORE unblinding"
print("\nOK. The next cell breaks the blind.")

## 6. Confirm A8 repairs are present in every cell (still blind)

In [ ]:
import numpy as np, json
from pathlib import Path
for cell in sorted(EXPECTED_CELLS):
    d = Path("/kaggle/working/probe-capacity-invariance") / "results/probes" / cell
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    assert "random_projector" in z.files, f"{cell}: re-run notebook 03 first"
    assert m.get("encoder_ckpts"), f"{cell}: provenance missing, re-run notebook 03"
    print(f"{cell:16s} OK  probe_train={m['probe_train_size']}")

## 7. Point of no return

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.hypotheses \
    --root results/probes --out results/hypotheses/hypothesis_report.json

## 8. Figures

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.figures \
    --root results/probes --out results/figures/real

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")